# Estimación de demanda y simulación de una fusión

Usted trabaja en un estudio de economía. Un cliente le pide estimar cuánto subirían los precios de los furgones livianos si se fusionan Grupo Bravado (marca Bravado: modelos Youga, Rumpo y Paradise) y Grupo Truffade (marcas Truffade, Ocelot y Ubermacht: modelos Nero, Adder, Jackal y Sentinel).

Los pasos a seguir son los siguientes:

1. Armar una base de ventas, precios y atributos, y describir el mercado: participaciones y concentración (HHI) antes y después de la fusión.
2. Estimar la demanda con un logit a partir de las participaciones de mercado, estimar por MCO y luego por instrumento de costos.
3. Proponer nidos para un nested logit.
4. Simular la fusión con competencia Bertrand-Nash en precios, usando la librería PyBLP.
5. Reportar las alzas de precios para distintos tamaños de mercado y explicar las razones de desvío entre las partes.


## 0. Preparación

En Google Colab hay que instalar PyBLP.

In [1]:
!pip install -q pyblp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 22.2 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import pyblp
import statsmodels.formula.api as smf

pyblp.options.verbose = False
pd.set_option("display.width", 120, "display.max_columns", 20)

URL = "https://raw.githubusercontent.com/joaquin-vmo/demanda_logit/main/furgones.csv"
b = pd.read_csv(URL)

## 1. Los datos

Usted reúne las ventas mensuales de furgones livianos nuevos en Chile entre enero de 2017 y diciembre de 2019, con su precio y sus atributos de catálogo.¹ Se define en este contexto el producto como: modelo + motor + combustible. Un mercado es un mes. La base tiene 611 filas versión-mes.

| Columna | Descripción |
|---|---|
| `mercado` | Mes (año-mes): cada mes es un mercado $t$ |
| `producto` | Versión $j$ (modelo + motor + combustible) |
| `modelo`, `marca`, `firma` | Modelo, marca y dueño de la marca |
| `origen` | Europeo o asiático |
| `ventas` | Unidades vendidas en el mes |
| `precio` | Millones de pesos de diciembre de 2019 |
| `cilindrada`, `potencia`, `peso`, `diesel` | Litros, kW, peso bruto en kg, 1 si es diésel |
| `tcr` | Tipo de cambio real (enero 2017 = 100) |
| `moneda`, `tipo_cambio` | Moneda de origen y tipo de cambio bilateral (no se usan) |

<small>¹ La base es semisintética: parte de datos reales de ventas, precios y atributos, con marcas y modelos renombrados, a los que se agregó un shock de costos y un shock de demanda conocidos. Por eso los resultados no describen a ninguna empresa real, y al final se pueden comparar las estimaciones con el valor verdadero.</small>

In [3]:
b.head(6).T

,0,1,2,3,4,5
mercado,2017-01,2017-01,2017-01,2017-01,2017-01,2017-01
producto,Annis Savestra 1.5 G,Benefactor Panto 1.5 D,Declasse Burrito 1.2 G,Bravado Paradise 1.3 D,Bravado Paradise 1.6 D,Bravado Rumpo 1.4 G
modelo,Annis Savestra,Benefactor Panto,Declasse Burrito,Bravado Paradise,Bravado Paradise,Bravado Rumpo
marca,Annis,Benefactor,Declasse,Bravado,Bravado,Bravado
firma,Annis,Benefactor,Declasse,Grupo Bravado,Grupo Bravado,Grupo Bravado
origen,asiatico,europeo,asiatico,europeo,europeo,europeo
ventas,14,12,148,20,16,59
precio,6.937,13.66,6.873,11.963,13.852,8.537
cilindrada,1.5,1.5,1.2,1.3,1.6,1.4
potencia,75.0,97.0,60.0,65.0,76.0,62.0


### Estadística descriptiva

Una versión vende en promedio 53 furgones al mes, pero con mucha dispersión. Los precios van de 4,4 a 21 millones. Los furgones europeos son diésel y más caros, los asiáticos, bencineros y más baratos.

In [4]:
b[["ventas", "precio", "cilindrada", "potencia", "peso", "diesel", "tcr"]].describe().T[["mean", "std", "min", "max"]].round(2)

,mean,std,min,max
ventas,52.58,78.25,1.00,654.00
precio,9.23,2.94,4.41,21.31
cilindrada,1.41,0.18,1.20,2.00
potencia,66.86,10.86,55.00,103.00
peso,1988.56,255.56,1680.00,2505.00
diesel,0.48,0.50,0.00,1.00
tcr,101.01,3.35,96.28,112.38


In [5]:
b["anio"] = b.mercado.str[:4]
b19 = b[b.anio == "2019"]
firmas = b19.groupby("firma").agg(
    marcas=("marca", lambda x: ", ".join(sorted(x.unique()))),
    versiones=("producto", "nunique"),
    origen=("origen", "first"),
    ventas=("ventas", "sum"),
    precio_medio=("precio", lambda x: np.average(x, weights=b19.ventas[x.index])),
)
firmas["participacion_%"] = 100 * firmas.ventas / firmas.ventas.sum()
firmas.sort_values("ventas", ascending=False).round(1)

,marcas,versiones,origen,ventas,precio_medio,participacion_%
firma,,,,,,
Grupo Truffade,"Ocelot, Truffade, Ubermacht",4,europeo,3258,10.8,43.6
Grupo Bravado,Bravado,5,europeo,1453,9.6,19.4
Declasse,Declasse,1,asiatico,1287,6.7,17.2
Penaud,Penaud,2,europeo,545,9.7,7.3
Karin,Karin,1,asiatico,291,5.7,3.9
Maibatsu,Maibatsu,2,asiatico,242,5.7,3.2
Annis,Annis,1,asiatico,133,6.6,1.8
Benefactor,Benefactor,1,europeo,128,14.1,1.7
Shitzu,Shitzu,1,asiatico,62,9.2,0.8


En 2019, Grupo Truffade es el líder con 44% de las ventas y Grupo Bravado el segundo con 19%. Juntos tendrían 63% del mercado. Ambos venden furgones europeos; los rivales más cercanos en tamaño son Declasse (asiático, 17%) y Penaud (europeo, 7%).

### Concentración: HHI antes y después

El índice de Herfindahl-Hirschman suma los cuadrados de las participaciones (en %):

$$HHI = \sum_f s_f^2, \qquad \Delta HHI = HHI^{post} - HHI^{pre} = 2\, s_{Bravado}\, s_{Truffade}.$$

El HHI post-fusión supone que las participaciones no cambian.

In [6]:
PARTES = ["Grupo Bravado", "Grupo Truffade"]
b["firma_post"] = b.firma.where(~b.firma.isin(PARTES), "Bravado+Truffade")

def hhi(ventas, firma):
    s = 100 * ventas.groupby(firma).sum() / ventas.sum()
    return (s ** 2).sum()

filas = []
for anio, g in b.groupby("anio"):
    pre, post = hhi(g.ventas, g.firma), hhi(g.ventas, g.firma_post)
    filas.append({"año": anio, "HHI pre": pre, "HHI post": post, "ΔHHI": post - pre})
pd.DataFrame(filas).set_index("año").round(0)

,HHI pre,HHI post,ΔHHI
año,,,
2017,2687.0,4398.0,1711.0
2018,2543.0,4220.0,1676.0
2019,2656.0,4348.0,1692.0


El mercado ya está concentrado (HHI ≈ 2.650) y la fusión lo sube a ≈ 4.350, con un aumento de ≈ 1.700 puntos todos los años. Es una fusión entre el primero y el segundo del mercado, así que merece un análisis de efectos.

## 2. Participaciones (Berry, 1994)

Cada consumidor $i$ en el mercado $t$ elige el furgón $j$ que le da más utilidad, o no compra ($j = 0$, como opción externa):

$$u_{ijt} = \underbrace{\alpha\, p_{jt} + x_j \beta + \xi_{jt}}_{\delta_{jt}} + \varepsilon_{ijt}, \qquad u_{i0t} = \varepsilon_{i0t}.$$

$\delta_{jt}$ es la utilidad media del producto, depende del precio $p_{jt}$, de los atributos observados $x_j$ y de una calidad no observada por el econometrista $\xi_{jt}$ (publicidad, red de concesionarios, reputación). Si $\varepsilon_{ijt}$ es valor extremo tipo I, la participación de mercado es

$$s_{jt} = \frac{e^{\delta_{jt}}}{1 + \sum_k e^{\delta_{kt}}}, \qquad s_{0t} = \frac{1}{1 + \sum_k e^{\delta_{kt}}}.$$

Berry (1994) invierte esta relación, con participaciones observadas se recupera la utilidad media, y la demanda queda como una regresión lineal:

$$\ln s_{jt} - \ln s_{0t} = \delta_{jt} = \alpha\, p_{jt} + x_j \beta + \xi_{jt}.$$

$\xi_{jt}$ es el papel del error. Así, con datos agregados (ventas por producto) se estima un modelo de elección discreta. Las elasticidades salen directo de $\alpha$:

$$\eta_{jj} = \alpha\, p_j (1 - s_j), \qquad \eta_{jk} = -\alpha\, p_k\, s_k \quad (k \neq j).$$

### El tamaño de mercado

Para calcular $s_{0t}$ hay que fijar el mercado potencial $M$, cuántos podrían haber comprado un furgón ese mes. Bjørnerstedt y Verboven (2016) usan un múltiplo fijo de las ventas ("the potential market is twice the average amount spent over the entire period") y revisan la sensibilidad a ese factor.

Aquí preeliminarmente usaremos $M$ fijo definido usando el mes de mayores ventas por un factor de 1,5.

In [7]:
FACTOR = 1.5
M = FACTOR * b.groupby("mercado").ventas.sum().max()
b["s"] = b.ventas / M
b["s0"] = 1 - b.groupby("mercado").s.transform("sum")
b["y"] = np.log(b.s) - np.log(b.s0)
print(f"M = {M:.0f} furgones al mes, s0 entre {b.s0.min():.2f} y {b.s0.max():.2f}")

M = 2474 furgones al mes, s0 entre 0.33 y 0.93


## 3. Logit

Se estima $\ln s_{jt} - \ln s_{0t}$ en el precio con tres conjuntos de controles. Todas incluyen efectos fijos de mes $\tau_t$, que absorben el ciclo, la estacionalidad y $s_{0t}$. Los errores tienen cluster por versión (23 clusters), porque los shocks de una versión se correlacionan en el tiempo. Convención de signo: aquí $\alpha < 0$.

1. Atributos + FE mes: cilindrada, potencia, peso y diésel.
2. Atributos + FE marca y mes: agrega lo común a cada marca (imagen, red de servicio).
3. FE versión y mes: un efecto fijo $\mu_j$ por versión absorbe todo lo fijo del furgón, observado e inobservable. Los atributos son fijos por versión, así que $\mu_j$ los absorbe (y no se estiman sus coeficientes).

Van Biesebroeck y Verboven (2026) escriben $-\alpha p$ con $\alpha > 0$.

In [8]:
cl = {"cov_type": "cluster", "cov_kwds": {"groups": pd.factorize(b.producto)[0]}}
ESPECIFICACIONES = {
    "(1) atributos + FE mes": "y ~ precio + cilindrada + potencia + peso + diesel + C(mercado)",
    "(2) atributos + FE marca y mes": "y ~ precio + cilindrada + potencia + peso + diesel + C(marca) + C(mercado)",
    "(3) FE versión y mes": "y ~ precio + C(producto) + C(mercado)",
}
filas = []
for nombre, formula in ESPECIFICACIONES.items():
    r = smf.ols(formula, b).fit(**cl)
    a = r.params["precio"]
    filas.append({"especificación": nombre, "α": a, "e.e.": r.bse["precio"],
                  "elasticidad mediana": np.median(a * b.precio * (1 - b.s)), "R2": r.rsquared})
mco = pd.DataFrame(filas).set_index("especificación")
mco.round(3)

,α,e.e.,elasticidad mediana,R2
especificación,,,,
(1) atributos + FE mes,-0.296,0.096,-2.662,0.481
(2) atributos + FE marca y mes,-0.460,0.109,-4.133,0.788
(3) FE versión y mes,-0.344,0.080,-3.090,0.876


Se elige la especificación (3), con efectos fijos de versión.

Todo surge de la ecuación (5) de Van Biesebroeck y Verboven (2026, §3.2.2), donde los controles "may include brand or product fixed effects". Con atributos solamente (1) o con marca (2), $\alpha$ se identifica comparando furgones distintos, y los más caros suelen tener mejor calidad no observada lo cual empuja $\alpha$ hacia cero.

Con (3) el efecto fijo de versión elimina esa parte fija de $\xi$, $\alpha$ se identifica solo con los cambios de precio de cada versión en el tiempo.

Pero el efecto fijo nos puede ayudar a estimar algo menos sesgado, pero no resuelve todo el problema.

## 4. ¿Por qué hacen falta instrumentos?

En (3) queda $\xi_{jt}$, la parte de la calidad no observada que cambia mes a mes, como una campaña de publicidad, una flota que compra, un concesionario nuevo. Las firmas conocen esos shocks y fijan precios pensando en ellos: "profit-maximizing manufacturers may incorporate unobserved product characteristics in their pricing decisions" (Van Biesebroeck y Verboven, 2026, §3.2.2).

Cuando la demanda de un furgón sube, su precio también sube. Entonces $\text{Cov}(p_{jt}, \xi_{jt}) > 0$ y el MCO confunde demanda alta con poca sensibilidad al precio: $\hat\alpha_{MCO}$ queda sesgado hacia cero, lo que "could entail a downward bias of the price coefficient". Berry (1994).

La solución es un instrumento $z_{jt}$ que cumpla dos condiciones:

- Relevancia: mueve el precio, $\text{Cov}(z_{jt}, p_{jt}) \neq 0$ (dados los efectos fijos).
- Exclusión: no se correlaciona con la demanda no observada, $E[\xi_{jt}\, z_{jt}] = 0$.

## 5. El instrumento de costos: tipo de cambio real × peso

Berry (1994, §4) propone los costos: "Cost variables that are excluded from $x_j$, such as input prices that vary across firms, are traditional instruments". Los costos mueven el precio pero no la utilidad del comprador.

Todos los furgones son importados, así que un peso chileno más débil (tipo de cambio real alto) los encarece. El problema es que el TCR es igual para todos los furgones del mes, y el efecto fijo de mes lo absorbe. V

an Biesebroeck y Verboven (2026, §3.2.2): los precios de insumos "do not imply meaningful variation if they are common across all products. Product-level variation can be introduced by interacting input prices with product characteristics".

El instrumento es

$$z_{jt} = \ln(TCR_t) \times \text{peso}_j / 1000.$$

Un alza del TCR encarece más a los furgones pesados (más acero, más flete). El supuesto de exclusión es que el TCR no responde a los shocks de demanda de un furgón en particular, y que lo que afecte a la demanda de todos los furgones en un mes lo absorbe el efecto fijo de mes.

Primera etapa: se regresa el precio en $z$ con los mismos efectos fijos. La regla práctica es $F > 10$, con un instrumento.

In [9]:
b["z"] = np.log(b.tcr) * b.peso / 1000
fs = smf.ols("precio ~ z + C(producto) + C(mercado)", b).fit(**cl)
print(f"coef. z = {fs.params['z']:.2f} (e.e. {fs.bse['z']:.2f}); F = {fs.tvalues['z'] ** 2:.1f}")

coef. z = 15.12 (e.e. 3.64); F = 17.3


El signo es el esperado: un TCR más alto sube el precio, y más en los pesados. Con $F = 17{,}3$ el instrumento pasa la regla.

### Logit IV con PyBLP

PyBLP (Conlon y Gortmaker, 2020) estima el logit con los efectos fijos absorbidos y el instrumento. Hay que nombrar las columnas como PyBLP espera: `market_ids`, `product_ids`, `firm_ids`, `prices`, `shares` y `demand_instruments0`. Con un instrumento y un precio endógeno, el modelo está exactamente identificado y equivale a MC2E. Además, PyBLP guarda todo lo que se necesita para simular la fusión.

In [10]:
d = b.rename(columns={"mercado": "market_ids", "producto": "product_ids", "firma": "firm_ids",
                      "precio": "prices", "s": "shares", "z": "demand_instruments0"})
d["clustering_ids"] = d.product_ids
X = pyblp.Formulation("0 + prices", absorb="C(product_ids) + C(market_ids)")
iv = pyblp.Problem(X, d).solve(method="1s", se_type="clustered")
a_iv, ee_iv = iv.beta[0, 0], iv.beta_se[0, 0]
elast = np.concatenate([np.diag(iv.compute_elasticities(market_id=t)) for t in iv.problem.unique_market_ids])
print(f"α IV = {a_iv:.3f} (e.e. {ee_iv:.3f}; IC 95% [{a_iv - 1.96 * ee_iv:.2f}, {a_iv + 1.96 * ee_iv:.2f}])")
print(f"elasticidad propia mediana = {np.median(elast):.2f}")

α IV = -0.650 (e.e. 0.176; IC 95% [-0.99, -0.31])
elasticidad propia mediana = -5.84


### Interpretación

- El coeficiente $\hat\alpha_{IV} = -0{,}65$: un millón de pesos más de precio baja la utilidad media en 0,65. Económicamente hablando lo que nos importar es la elasticidad, se lee como: si una versión sube su precio 1%, sus ventas caen cerca de 5,8%. Ninguna firma fija precios donde la demanda es inelástica, así que una elasticidad mayor que 1 en valor absoluto es una primera prueba de coherencia.
- Al instrumentar, $\alpha$ pasa de −0,34 a −0,65. Es la dirección que predicha del sesgo en las discusiones anteriores. El IV usa solo la parte del precio que mueven los costos, que no tiene que ver con $\xi$.

In [11]:
pd.DataFrame({"α": [mco.loc["(3) FE versión y mes", "α"], a_iv],
              "e.e.": [mco.loc["(3) FE versión y mes", "e.e."], ee_iv],
              "elasticidad mediana": [mco.loc["(3) FE versión y mes", "elasticidad mediana"], np.median(elast)]},
             index=["Logit MCO", "Logit IV (TCR × peso)"]).round(3)

,α,e.e.,elasticidad mediana
Logit MCO,-0.344,0.080,-3.090
Logit IV (TCR × peso),-0.650,0.176,-5.838


## 6. Propuesta de nested logit

El logit tiene un defecto visto en clases, la independencia de alternativas irrelevantes. Si sube el precio de un furgón, quienes dejan de comprarlo se reparten entre los demás en proporción a sus participaciones, sin importar cuán parecidos sean. Es probable que un comprador de un furgón europeo diésel se cambie a otro europeo diésel antes que a uno asiático bencinero.

El nested logit agrupa los productos en nidos $g$ y permite que los consumidores sustituyan más dentro del nido. Se muestra que también se invierte en una ecuación lineal:

$$\ln s_{jt} - \ln s_{0t} = \alpha\, p_{jt} + \rho \ln s_{j|g,t} + \mu_j + \tau_t + \xi_{jt},$$

donde $s_{j|g}$ es la participación de $j$ dentro de su nido y $\rho \in [0, 1)$ mide la correlación de preferencias dentro del nido. Con $\rho = 0$ se vuelve al logit, con $\rho \to 1$, solo se sustituye dentro del nido.

$\ln s_{j|g}$ también es endógena (contiene $\xi_{jt}$), así que necesita sus propios instrumentos. Se usan dos que cambian en el tiempo: el número de otras versiones en el nido (entrada y salida de productos) y la suma del instrumento de costos de esas versiones (cuando suben los costos de los rivales cercanos, la participación dentro del nido sube).

Se prueban cinco nidos, origen (europeo o asiático), combustible, tamaño (peso bruto bajo o sobre 2.000 kg), origen x combustible y modelo (las versiones de un mismo modelo). Se deja que $\rho$ tome cualquier valor entre −1 y 1 para ver qué dicen los datos.

In [12]:
b["tamano"] = np.where(b.peso < 2000, "pequeño", "grande")
b["combustible"] = np.where(b.diesel == 1, "diesel", "bencina")
b["origen_comb"] = b.origen + "-" + b.combustible
filas = []
for nido in ["origen", "combustible", "tamano", "origen_comb", "modelo"]:
    g = b.groupby(["mercado", nido])
    b["ln_sjg"] = np.log(b.ventas / g.ventas.transform("sum"))
    b["n_nido"] = g.producto.transform("size") - 1   # otras versiones en el nido
    b["z_nido"] = g.z.transform("sum") - b.z        # shocks de costos de esas versiones
    F = smf.ols("ln_sjg ~ z + n_nido + z_nido + C(producto) + C(mercado)", b).fit(**cl).wald_test(
        "z = 0, n_nido = 0, z_nido = 0", scalar=True, use_f=True).statistic
    dn = d.assign(nesting_ids=b[nido], demand_instruments1=b.n_nido, demand_instruments2=b.z_nido)
    r = pyblp.Problem(X, dn).solve(rho=0.3, rho_bounds=(-0.99, 0.99), method="1s", se_type="clustered")
    filas.append({"nido": nido, "F ln s_j|g": F, "α": r.beta[0, 0], "e.e. α": r.beta_se[0, 0],
                  "ρ": r.rho[0, 0], "e.e. ρ": r.rho_se[0, 0]})
pd.DataFrame(filas).set_index("nido").round(3)

,F ln s_j|g,α,e.e. α,ρ,e.e. ρ
nido,,,,,
origen,2.562,-0.675,0.334,-0.055,0.595
combustible,2.833,-0.795,0.245,-0.246,0.408
tamano,1.072,-0.561,0.258,0.054,0.685
origen_comb,4.939,-0.689,0.205,-0.027,0.299
modelo,7.155,-0.749,0.213,-0.326,0.298


Ningún nido funciona. En todos, $\hat\rho$ es cercano a cero o negativo (un $\rho < 0$ no es consistente con maximización de utilidad) y nunca significativo. Los instrumentos de $\ln s_{j|g}$ son débiles ($F$ entre 1 y 7).

Aquí es importante también la especificacioón que estemos estimando, por ejemplo usamos efecto fijo de versión, por lo que $\rho$ solo se identifica con cambios en el tiempo (entrada y salida de versiones, costos de los rivales del nido), y en tres años de un mercado con 23 versiones hay poca variación de ese tipo.

Que los nidos no den es un resultado posible y no es un problema en si, los nidos son siempre estructuras impuestas por el econometrista.

## 7. Simulación de la fusión: Bertrand-Nash

Suponemos que cada firma $f$ elige los precios de sus productos $\mathcal{J}_f$ para maximizar su utilidad, dados los precios de las demás (Bertrand-Nash), con costo marginal constante $c_j$:

$$\max_{p_j,\, j \in \mathcal{J}_f} \sum_{j \in \mathcal{J}_f} (p_j - c_j)\, M\, s_j(p).$$

La condición de primer orden de cada producto es

$$s_j + \sum_{k \in \mathcal{J}_f} (p_k - c_k) \frac{\partial s_k}{\partial p_j} = 0 \quad\Longleftrightarrow\quad s + \Delta (p - c) = 0, \qquad \Delta_{jk} = \Omega_{jk}\, \frac{\partial s_k}{\partial p_j},$$

donde $\Omega_{jk} = 1$ si $j$ y $k$ son de la misma firma. La simulación tiene dos pasos (Bjørnerstedt y Verboven, 2016):

1. Recuperar los costos. Con la demanda estimada y los precios observados, la condición de primer orden entrega los costos marginales: $c = p + \Delta^{-1} s$. No hace falta observar costos.
2. Nuevo equilibrio. La fusión cambia la matriz de propiedad $\Omega$: ahora los productos de Bravado y Truffade son de la misma firma. Se resuelven las condiciones de primer orden con la nueva $\Omega$ y los mismos costos.

La fusión sube precios porque la firma fusionada **internaliza** que parte de las ventas que pierde un producto al subir su precio se van a los productos de su socio. Se supone que la fusión no genera eficiencias (los costos no cambian) y que la demanda es la misma.

En PyBLP son dos líneas: `compute_costs()` hace el paso 1 y `compute_prices(firm_ids=...)` el paso 2 (resuelve el punto fijo de Morrow y Skerlos, 2011). Como $M$ es un supuesto, la función reestima con cada tamaño de mercado. Se reportan las alzas de 2019, ponderadas por ventas.

In [13]:
FUSION = {"Grupo Bravado": "Bravado+Truffade", "Grupo Truffade": "Bravado+Truffade"}
en_2019 = (d.market_ids.str[:4] == "2019").to_numpy()
ventas_max = d.groupby("market_ids").ventas.sum().max()

def fusion(factor):
    # M = factor × ventas del mes máximo; reestima, recupera costos y resuelve el nuevo equilibrio
    r = pyblp.Problem(X, d.assign(shares=d.ventas / (factor * ventas_max))).solve(method="1s", se_type="clustered")
    costos = r.compute_costs().flatten()                                       # paso 1
    p_post = r.compute_prices(firm_ids=d.firm_ids.replace(FUSION), costs=costos).flatten()  # paso 2
    return r, costos, p_post

def alza(p_post):
    # alza % de 2019 ponderada por ventas, para cada parte y para los rivales
    dp = pd.Series(100 * (p_post / d.prices - 1))[en_2019]
    grupo = d.firm_ids[en_2019].where(d.firm_ids[en_2019].isin(FUSION), "rivales")
    return dp.groupby(grupo).apply(lambda x: np.average(x, weights=d.ventas[x.index]))

filas = []
for factor in [1.25, 1.5, 2, 3, 5]:
    r, costos, p_post = fusion(factor)
    filas.append({"M = factor × ventas máx.": factor, "s0 mínimo": 1 - 1 / factor,
                  "elasticidad mediana": np.median(np.concatenate([np.diag(r.compute_elasticities(market_id=t)) for t in r.problem.unique_market_ids])),
                  "costos < 0": (costos < 0).sum(), **alza(p_post)})
pd.DataFrame(filas).set_index("M = factor × ventas máx.").round(2)

,s0 mínimo,elasticidad mediana,costos < 0,Grupo Bravado,Grupo Truffade,rivales
M = factor × ventas máx.,,,,,,
1.25,0.20,-5.79,0,3.26,1.14,0.02
1.50,0.33,-5.84,0,2.58,0.92,0.01
2.00,0.50,-5.89,0,1.82,0.66,0.00
3.00,0.67,-5.97,0,1.15,0.43,0.00
5.00,0.80,-6.05,0,0.66,0.25,0.00


Las columnas de firmas son el alza porcentual promedio de sus precios en 2019. Ningún costo marginal recuperado es negativo, una prueba básica de que la demanda estimada es coherente con los precios.

El alza cae con $M$. Lo que cambia son las participaciones, mientras más grande el meracdo potencial, $s_j = q_j / M$ es más chica para todos y la opción externa es más grande. Cuando la firma fusionada sube un precio, la mayoría de los clientes que pierde se van a "no comprar" y pocos a su socio. La fusión internaliza menos y el alza es menor.

Bravado sube más que Truffade. En el logit, el margen absoluto de una firma multiproducto es el mismo para todos sus productos,

$$p_j - c_j = \frac{1}{-\alpha\,(1 - S_f)}, \qquad S_f = \sum_{k \in \mathcal{J}_f} s_k.$$

El margen depende solo de la participación total de la firma. Tras la fusión, ambas partes pasan a tener el margen de la firma fusionada, con $S_{Bravado} + S_{Truffade}$. Para Truffade, que ya tenía una participación grande, el cambio es chico, para Bravado, que pasa de su participación propia a la del conjunto, el cambio es grande.

In [14]:
r, costos, p_post = fusion(FACTOR)
m19 = d[en_2019].assign(margen=(d.prices - costos)[en_2019], margen_post=(p_post - costos)[en_2019],
                        alza_pct=100 * (p_post / d.prices - 1)[en_2019])
S_f = m19.groupby(["market_ids", "firm_ids"]).shares.transform("sum")
m19["1/(-α(1-S_f))"] = 1 / (-r.beta[0, 0] * (1 - S_f))
cols = ["prices", "margen", "1/(-α(1-S_f))", "margen_post", "alza_pct"]
m19[m19.firm_ids.isin(FUSION)].groupby(["firm_ids", "product_ids"])[cols].mean().round(2)

prices  margen  1/(-α(1-S_f))  margen_post  alza_pct
firm_ids       product_ids                                                                   
Grupo Bravado  Bravado Paradise 1.3 D     12.24    1.62           1.62         1.82      1.72
               Bravado Paradise 1.6 D     13.69    1.62           1.62         1.82      1.53
               Bravado Rumpo 1.4 G         7.88    1.62           1.62         1.82      2.61
               Bravado Youga 1.2 D        10.20    1.62           1.62         1.82      2.01
               Bravado Youga 1.4 G         8.59    1.62           1.62         1.82      2.43
Grupo Truffade Ocelot Jackal 1.6 D        10.93    1.74           1.74         1.82      0.76
               Truffade Nero 1.5 D        18.51    1.69           1.69         1.77      0.40
               Truffade Nero 1.6 D        11.38    1.74           1.74         1.82      0.73
               Ubermacht Sentinel 1.6 D   12.73    1.65           1.65         1.71      0.50

## 8. Razones de desvío asimétricas

La razón de desvío de $j$ a $k$ es la fracción de las ventas que pierde $j$ al subir su precio que se van a $k$:

$$D_{jk} = -\frac{\partial s_k / \partial p_j}{\partial s_j / \partial p_j} = \frac{s_k}{1 - s_j} \quad \text{(logit)}.$$

Para pasar de productos a firmas se suman los $k$ de la firma de destino y se promedian los $j$ de la firma de origen, ponderando por ventas. PyBLP las calcula con `compute_diversion_ratios`.

In [15]:
def desvio(resultado, desde, hacia):
    # desvío promedio 2019 de las ventas de 'desde' hacia 'hacia', ponderado por ventas
    num = den = 0
    for t in d.market_ids[en_2019].unique():
        D = resultado.compute_diversion_ratios(market_id=t)  # D[j, k]: de j a k; diagonal: a no comprar
        f, q = d.firm_ids[d.market_ids == t].to_numpy(), d.ventas[d.market_ids == t].to_numpy()
        de = f == desde
        x = np.diag(D)[de] if hacia == "no comprar" else D[np.ix_(de, f == hacia)].sum(1)
        num, den = num + q[de] @ x, den + q[de].sum()
    return 100 * num / den

filas = {}
for factor in [1.5, 3]:
    r = fusion(factor)[0]
    filas[f"M = {factor} × ventas máx."] = {
        "Bravado → Truffade": desvio(r, "Grupo Bravado", "Grupo Truffade"),
        "Truffade → Bravado": desvio(r, "Grupo Truffade", "Grupo Bravado"),
        "Bravado → no comprar": desvio(r, "Grupo Bravado", "no comprar"),
        "Truffade → no comprar": desvio(r, "Grupo Truffade", "no comprar")}
pd.DataFrame(filas).round(1)

,M = 1.5 × ventas máx.,M = 3 × ventas máx.
Bravado → Truffade,13.0,6.5
Truffade → Bravado,6.5,3.0
Bravado → no comprar,72.8,86.5
Truffade → no comprar,75.8,88.6


El desvío de Bravado a Truffade es cerca del doble que el de Truffade a Bravado. En el logit, $D_{jk} = s_k/(1 - s_j)$, lo que recibe un destino es proporcional a su participación. Truffade vende más del doble que Bravado, así que recibe una fracción mayor de los clientes que pierde Bravado que al revés.

Esta asimetría es la misma que explica las alzas: la fusión hace que Bravado "recupere" a través de Truffade buena parte de lo que pierde al subir su precio, mientras que Truffade recupera poco a través de Bravado. Por eso el alza se concentra en los precios de Bravado.

## 9. Conclusión

- **Concentración.** La fusión une al primero (44%) y al segundo (19%) del mercado de furgones livianos. El HHI sube de ≈ 2.650 a ≈ 4.350.
- **Demanda.** El logit por MCO subestima la sensibilidad al precio porque los precios responden a la demanda no observada. Instrumentando con el tipo de cambio real × peso ($F = 17{,}3$), $\alpha$ pasa de −0,34 a −0,65 y la elasticidad propia mediana se acerca a −5,8. Los nested logit por origen, combustible, tamaño o modelo no identifican $\rho$, así que se usa el logit IV.
- **Efectos.** Con competencia Bertrand-Nash y sin eficiencias, la fusión sube los precios de Bravado entre 0,7% y 3,3% y los de Truffade entre 0,3% y 1,1%, según el tamaño de mercado; los rivales casi no cambian sus precios. El escenario base ($M = 1{,}5 \times$ las ventas máximas) da 2,6% para Bravado y 0,9% para Truffade.
- **Por qué esa asimetría.** Truffade es más del doble de grande: recibe una fracción mayor de los clientes que pierde Bravado ($D \approx$ 13% frente a 6,5% en el escenario base), así que la fusión cambia más los incentivos de precio de Bravado.
- **Advertencias.** Los resultados dependen del tamaño de mercado supuesto, del patrón de sustitución del logit (que depende solo de las participaciones) y de que la fusión no genere eficiencias. Eficiencias de costos reducirían las alzas.

## Apéndice

La base es semisintética. Parte de datos reales de ventas, precios y atributos de furgones livianos en Chile (2017–2019), con marcas y modelos renombrados. Se agregaron dos shocks conocidos (`datos/construir_base.py`):

- Costos: $\Delta p_{jt} = 1{,}75 \times \ln(TCR_t / 100) \times \text{peso}_j / 1000 \times p_{jt} / 2$.
- Demanda: $\xi_{jt} \sim N(0, 0{,}2)$, que además sube el precio en $1{,}5\, \xi_{jt}$ millones (la endogeneidad).

Las ventas responden a ambos con $\alpha = -0{,}6$: $q_{jt} \to q_{jt}\, e^{-0{,}6\, \Delta p_{jt} + \xi_{jt}}$. El $\alpha$ verdadero es −0,6.

## Referencias

- Berry, S. (1994). Estimating discrete-choice models of product differentiation. *RAND Journal of Economics* 25(2): 242–262.
- Bjørnerstedt, J. y Verboven, F. (2016). Does merger simulation work? Evidence from the Swedish analgesics market. *AEJ: Applied Economics* 8(3): 125–164.
- Conlon, C. y Gortmaker, J. (2020). Best practices for differentiated products demand estimation with PyBLP. *RAND Journal of Economics* 51(4): 1108–1161.
- Van Biesebroeck, J. y Verboven, F. (2026). Demand, competition, and public policy in the automobile industry. *Journal of Economic Literature* 64(3): 984–1033.